# Scraping de YouTube para moderacion de contenido

**Politica y farandula peruana**

**Objetivo:** recolectar metadatos y transcripciones publicas sin usar APIs ni llaves externas. El cuaderno sigue el estilo de los ejemplos de clase: instalacion, exploracion, scraping, organizacion en tablas y almacenamiento local.

**Flujo:**
1. Definir canales semilla.
2. Buscar canales candidatos con Selenium sobre la web publica.
3. Extraer videos recientes con `yt-dlp` sin descargar video.
4. Descargar subtitulos/transcripciones publicas con `yt-dlp`.
5. Guardar un archivo `jsonl` compatible con el cuaderno de limpieza.

## 1. Instalacion de librerias

Estas librerias se usan de manera local. No se usa YouTube Data API, Google API ni servicios de LLM.

In [3]:
# Instalación de librerías del proyecto
# youtube-transcript-api es un scraper sin API key (fallback para transcripciones)
!pip install -q pandas selenium beautifulsoup4 webdriver-manager yt-dlp tqdm youtube-transcript-api


In [4]:
from pathlib import Path
import hashlib
import json
import re
import ssl
import time
import urllib3
from urllib.parse import quote_plus

# ── Parche SSL para proxy corporativo con certificado autofirmado ─────────────
# El proxy intercepta TLS y presenta su propio cert; se deshabilita la
# verificación estricta sólo para esta sesión del notebook.
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'datos' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Directorio de trabajo:', ROOT)
print('Salida raw:', RAW_DIR)
print('SSL check: deshabilitado (proxy corporativo)')


Directorio de trabajo: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4
Salida raw: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw
SSL check: deshabilitado (proxy corporativo)


## 2. Canales semilla

La lista inicial no es un ranking cerrado. Sirve para iniciar el corpus y debe revisarse manualmente segun disponibilidad de transcripciones, calidad de audio y pertinencia del contenido.

In [5]:
# registro_linguistico:
#   formal      → lenguaje periodístico o académico estándar
#   informal    → coloquial pero inteligible; mezcla registros
#   coloquial   → jerga, peruanismos, humor, pace rápida
#   incorrecto  → groserías, habla popular, errores gramaticales intencionados

canales_semilla = pd.DataFrame([
    # ── Política / periodismo ─────────────────────────────────────────────────
    {'nombre': 'Marco Sifuentes / Ocram',          'categoria': 'politica_analisis',    'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ocram',                          'nota': 'La Encerrona; análisis político con tono irónico'},
    {'nombre': 'El diario de Curwen',              'categoria': 'politica_opinion',     'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@curwen',                         'nota': 'opinión política con sarcasmo y lenguaje muy coloquial peruano'},
    {'nombre': 'Sin Guion con Rosa Maria Palacios','categoria': 'politica_periodismo',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@singuionlr',                     'nota': 'periodismo de opinión y entrevistas'},
    {'nombre': 'RPP Noticias',                     'categoria': 'politica_actualidad',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@RPPNoticias',                    'nota': 'noticias y entrevistas; lenguaje periodístico estándar'},
    {'nombre': 'Exitosa Noticias',                 'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@exitosanoticias',                'nota': 'noticias, entrevistas y opinión; tono más popular que RPP'},
    {'nombre': 'Willax Television',                'categoria': 'politica_opinion',     'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@WillaxTV',                       'nota': 'programas de opinión política; lenguaje a veces confrontacional'},
    {'nombre': 'Canal N',                          'categoria': 'politica_actualidad',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@canaln',                         'nota': 'noticias y entrevistas; canal de cable informativo'},
    {'nombre': 'ATV Noticias',                     'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ATVNoticias',                    'nota': 'noticias nacionales; lenguaje más popular que canal N'},
    {'nombre': 'Latina Noticias',                  'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@latinanoticias',                 'nota': 'noticias y magazine; amplia cobertura nacional'},
    {'nombre': 'Panamericana TV',                  'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@PanamericanaTV',                 'nota': 'noticias y entretenimiento; cobertura regional fuera de Lima'},

    # ── Humor / streaming / lenguaje coloquial e incorrecto ───────────────────
    {'nombre': 'Hablando Huevadas',                'categoria': 'humor_streaming',      'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@HablandoHuevadasOficial',        'nota': 'humor adulto; groserías, peruanismos; referente para lenguaje ofensivo'},
    {'nombre': 'Todo Good',                        'categoria': 'streaming_opinion',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@todogoodpe',                     'nota': 'conversación, humor, invitados y coyuntura; mezcla registros'},
    {'nombre': 'Goblinciano',                      'categoria': 'streaming_opinion',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@Goblinciano',                    'nota': 'streaming largo; memes, opinión peruana, lenguaje muy coloquial'},
    {'nombre': 'El Cacas',                         'categoria': 'humor_streaming',      'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@ElCacas',                        'nota': 'humor y retos; jerga peruana urbana contemporánea'},
    {'nombre': 'Negro Fuertes',                    'categoria': 'humor_comedia',        'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@NegroFuertes',                   'nota': 'comedia adulta; lenguaje soez y coloquial; referente de humor peruano popular'},
    {'nombre': 'Jason Qqq',                        'categoria': 'humor_streaming',      'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@JasonQqqOficial',                'nota': 'reacciones y gaming; lenguaje muy informal y groserías frecuentes'},
    {'nombre': 'La Cotorrisa Peru',                'categoria': 'humor_podcast',        'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@LaCotorrisaPeru',                'nota': 'podcast de humor; adaptación peruana; lenguaje coloquial'},

    # ── Farándula / espectáculos ───────────────────────────────────────────────
    {'nombre': 'Magaly TV La Firme',               'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@MagalyTVLaFirme',               'nota': 'farándula; conflicto público y lenguaje de espectáculos'},
    {'nombre': 'Amor y Fuego',                     'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@AmoryFuego',                    'nota': 'farándula y comentarios de entretenimiento'},
    {'nombre': 'America Hoy',                      'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@americahoytv',                  'nota': 'magazine y entretenimiento; verificar disponibilidad de videos'},
    {'nombre': 'Instarandula',                     'categoria': 'farandula_digital',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@Instarandula',                  'nota': 'Samuel Suárez; farándula digital; lenguaje coloquial con frases de peruanismos'},
    {'nombre': 'El Popular',                       'categoria': 'farandula_digital',    'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ElPopularPeru',                 'nota': 'tabloid popular; espectáculos y farándula con lenguaje accesible'},

    # ── Deportes ──────────────────────────────────────────────────────────────
    {'nombre': 'Nico Moschella',                   'categoria': 'deportes_informal',    'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@NicoMoschella',                 'nota': 'comentarista deportivo; lenguaje muy informal, apodos y jerga futbolera'},
    {'nombre': 'Libero Deportes',                  'categoria': 'deportes',             'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@LiberoOficial',                 'nota': 'diario Libero; cobertura deportiva con lenguaje popular'},
    {'nombre': 'Depor',                            'categoria': 'deportes',             'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@DeporPeru',                     'nota': 'portal deportivo El Comercio; cobertura variada'},

    # ── Viajes / gastronomía ──────────────────────────────────────────────────
    {'nombre': 'Misias pero viajeras',             'categoria': 'viajes',               'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/c/Misiasperoviajeras',           'nota': 'archivo de viajes; canal inactivo pero episodios útiles'},
    {'nombre': 'Buen Viaje',                       'categoria': 'viajes',               'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/c/BuenViajePe',                 'nota': 'viajes por Perú con lenguaje descriptivo'},
    {'nombre': 'Viaja y Prueba',                   'categoria': 'viajes_gastronomia',   'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ViajayPrueba',                 'nota': 'viajes y gastronomía por Perú'},
    {'nombre': 'Cocinando con la Patty',           'categoria': 'gastronomia',          'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@CocinandoConLaPatty',          'nota': 'cocina peruana; lenguaje cotidiano regional'},
])

candidatos_por_verificar = pd.DataFrame([
    {'nombre': 'Phillip Butters',        'categoria': 'politica_opinion',  'consulta': 'Phillip Butters YouTube Peru canal oficial',        'nota': 'opinión política polémica; lenguaje directo y confrontacional; verificar canal activo'},
    {'nombre': 'Beto Ortiz',             'categoria': 'periodismo_opinion', 'consulta': 'Beto Ortiz YouTube Peru canal oficial entrevistas', 'nota': 'periodismo de investigación e entrevistas; verificar canal oficial'},
    {'nombre': 'Doble Merito',           'categoria': 'humor_streaming',    'consulta': 'Doble Merito YouTube Peru canal streaming',         'nota': 'humor peruano urbano; verificar URL y actividad'},
    {'nombre': 'Peluche Oficial',        'categoria': 'humor_comedia',      'consulta': 'Peluche youtuber peruano canal oficial',            'nota': 'comedia peruana; verificar canal y disponibilidad de subtítulos'},
    {'nombre': 'Renzo Reggiardo',        'categoria': 'politica_opinion',   'consulta': 'Renzo Reggiardo YouTube Peru canal oficial',        'nota': 'opinión y política; verificar si sube contenido regularmente'},
    {'nombre': 'DoblexD Peru',           'categoria': 'humor_gaming',       'consulta': 'DoblexD YouTube Peru gaming humor peruano',         'nota': 'gaming y humor; jerga gamer peruana; verificar URL'},
    {'nombre': 'Combate / EEG Peru',     'categoria': 'entretenimiento_tv', 'consulta': 'Combate EEG Peru YouTube canal oficial',            'nota': 'reality; lenguaje juvenil peruano; verificar canal actualizado'},
    {'nombre': 'La Paisana Jacinta',     'categoria': 'humor_regional',     'consulta': 'Paisana Jacinta YouTube Peru canal oficial',        'nota': 'humor regional peruano; acento y modismos andinos; verificar disponibilidad'},
])

canales_semilla.to_csv(RAW_DIR / 'canales_semilla.csv', index=False)
candidatos_por_verificar.to_csv(RAW_DIR / 'candidatos_por_verificar.csv', index=False)

print(f'Canales semilla      : {len(canales_semilla)}')
print(f'Candidatos a verificar: {len(candidatos_por_verificar)}')
print()
print(canales_semilla.groupby(['categoria', 'registro_linguistico']).size().to_string())


Canales semilla      : 29
Candidatos a verificar: 8

categoria            registro_linguistico
deportes             informal                2
deportes_informal    incorrecto              1
farandula            informal                3
farandula_digital    coloquial               1
                     informal                1
gastronomia          coloquial               1
humor_comedia        incorrecto              1
humor_podcast        coloquial               1
humor_streaming      coloquial               1
                     incorrecto              2
politica_actualidad  formal                  2
                     informal                4
politica_analisis    informal                1
politica_opinion     coloquial               1
                     informal                1
politica_periodismo  formal                  1
streaming_opinion    coloquial               2
viajes               coloquial               1
                     informal                1
viajes_gastr

## 3. Busqueda web con Selenium

Esta seccion reproduce la logica de scraping vista en clase: abrir navegador, cargar una pagina, obtener HTML y parsear enlaces. Si ya se tienen URLs verificadas, esta parte puede omitirse.

In [6]:
UA = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/124.0.0.0 Safari/537.36'
)


def iniciar_driver(headless=True):
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1400,1000')
    options.add_argument('--lang=es-PE')
    options.add_argument(f'--user-agent={UA}')
    # Ocultar señales de automatización
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    # Eliminar propiedad webdriver del navegador
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'}
    )
    return driver


def buscar_canales_youtube(consulta, n_scrolls=2, headless=True):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = iniciar_driver(headless=headless)
    try:
        # &sp=EgIQAg%3D%3D filtra resultados por tipo 'Canal'
        url = ('https://www.youtube.com/results?search_query='
               + quote_plus(consulta) + '&sp=EgIQAg%3D%3D')
        driver.get(url)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, 'ytd-channel-renderer'))
            )
        except Exception:
            time.sleep(4)  # fallback si ytd-channel-renderer no aparece
        for _ in range(n_scrolls):
            driver.execute_script('window.scrollTo(0, document.documentElement.scrollHeight);')
            time.sleep(2)
        html = driver.page_source
    finally:
        driver.quit()

    soup = BeautifulSoup(html, 'html.parser')
    filas = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        texto = a.get_text(' ', strip=True)
        if ('/@' in href or '/channel/' in href or '/c/' in href) and texto:
            if href.startswith('/'):
                href = 'https://www.youtube.com' + href
            filas.append({'consulta': consulta, 'nombre': texto, 'url': href.split('?')[0]})
    return pd.DataFrame(filas).drop_duplicates('url')


# Consultas agrupadas por tipo de contenido y registro lingüístico objetivo
consultas = [
    # Política / periodismo formal
    'noticias politica peru canal youtube',
    'periodismo opinion peru youtube canal',
    'RPP Exitosa Canal N ATV Latina Peru noticias',

    # Política / opinión informal
    'Marco Sifuentes Ocram La Encerrona YouTube Peru',
    'Curwen diario YouTube Peru canal opinion',
    'Willax Phillip Butters Beto Ortiz YouTube Peru opinion',
    'analisis politico peruano youtube podcast',

    # Humor / streaming / lenguaje coloquial e incorrecto
    'Hablando Huevadas YouTube Peru canal oficial humor',
    'Goblinciano YouTube Peru canal streaming',
    'El Cacas youtuber peruano canal oficial',
    'Negro Fuertes comedia peruana YouTube canal',
    'Jason Qqq YouTube Peru canal reacciones',
    'Todo Good streaming Peru youtube canal',
    'youtubers peruanos humor adulto lenguaje coloquial',
    'comedia peru youtube canal popular 2024',

    # Farándula / espectáculos
    'Magaly Amor y Fuego America Hoy YouTube Peru farandula',
    'Instarandula Samuel Suarez YouTube canal oficial',
    'El Popular farandula Peru youtube canal',
    'chismes espectaculos peru youtube canal 2024',

    # Deportes lenguaje informal
    'Nico Moschella futbol peru YouTube canal',
    'Libero Depor futbol peruano YouTube canal comentarios',
    'comentaristas deportivos peruanos youtube canal informal',

    # Viajes / gastronomía / vloggers
    'youtubers peruanos viajes gastronomia Peru canal',
    'Misias Buen Viaje Viaja y Prueba YouTube Peru',
    'vloggers Peru 2024 youtube canal lifestyle',

    # Humor regional / cultura popular
    'humor regional peruano youtube canal serrano costeno',
    'cultura popular peruana youtube canal entretenimiento',
]


In [7]:
# Ejecutar esta celda si se desea descubrir nuevos canales desde la web publica.
# Puede tardar por la carga dinamica de YouTube.

ejecutar_busqueda = False

if ejecutar_busqueda:
    candidatos = []
    for consulta in tqdm(consultas):
        candidatos.append(buscar_canales_youtube(consulta, n_scrolls=2, headless=True))
    canales_candidatos = pd.concat(candidatos, ignore_index=True).drop_duplicates('url')
    canales_candidatos.to_csv(RAW_DIR / 'canales_candidatos_scraping.csv', index=False)
else:
    canales_candidatos = canales_semilla.copy()

canales_candidatos.head(20)

,nombre,categoria,registro_linguistico,url,nota
0,Marco Sifuentes / Ocram,politica_analisis,informal,https://www.youtube.com/@ocram,La Encerrona; análisis político con tono irónico
1,El diario de Curwen,politica_opinion,coloquial,https://www.youtube.com/@curwen,opinión política con sarcasmo y lenguaje muy c...
2,Sin Guion con Rosa Maria Palacios,politica_periodismo,formal,https://www.youtube.com/@singuionlr,periodismo de opinión y entrevistas
3,RPP Noticias,politica_actualidad,formal,https://www.youtube.com/@RPPNoticias,noticias y entrevistas; lenguaje periodístico ...
4,Exitosa Noticias,politica_actualidad,informal,https://www.youtube.com/@exitosanoticias,"noticias, entrevistas y opinión; tono más popu..."
5,Willax Television,politica_opinion,informal,https://www.youtube.com/@WillaxTV,programas de opinión política; lenguaje a vece...
6,Canal N,politica_actualidad,formal,https://www.youtube.com/@canaln,noticias y entrevistas; canal de cable informa...
7,ATV Noticias,politica_actualidad,informal,https://www.youtube.com/@ATVNoticias,noticias nacionales; lenguaje más popular que ...
8,Latina Noticias,politica_actualidad,informal,https://www.youtube.com/@latinanoticias,noticias y magazine; amplia cobertura nacional
9,Panamericana TV,politica_actualidad,informal,https://www.youtube.com/@PanamericanaTV,noticias y entretenimiento; cobertura regional...


## 4. Extraccion de videos con yt-dlp

`yt-dlp` permite leer metadatos publicos y subtitulos sin usar una API key. Para mantener el corpus controlado, se limita el numero de videos por canal.

In [8]:
import yt_dlp

YT_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    )
}

# Opciones base compartidas entre listar y descargar
_YT_BASE_OPTS = {
    'quiet': True,
    'no_warnings': True,
    'ignoreerrors': True,
    'nocheckcertificate': True,   # fix: proxy corporativo con cert autofirmado
    'extractor_retries': 3,
    'http_headers': YT_HEADERS,
    'sleep_interval': 1,
    'max_sleep_interval': 3,
}


def _normalizar_url_canal(channel_url):
    """Garantiza que la URL apunte a la pestaña /videos del canal."""
    base = channel_url.rstrip('/')
    for sufijo in ('/videos', '/streams', '/shorts', '/playlists'):
        if base.endswith(sufijo):
            base = base[: -len(sufijo)]
    return base + '/videos'


def listar_videos_canal(channel_url, max_videos=15):
    url_videos = _normalizar_url_canal(channel_url)
    opciones = {
        **_YT_BASE_OPTS,
        'extract_flat': 'in_playlist',
        'playlist_items': f'1:{max_videos}',
        'skip_download': True,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            info = ydl.extract_info(url_videos, download=False)
    except Exception as exc:
        print(f'  ✗ Error extrayendo canal {channel_url}: {exc}')
        return []

    if not info:
        return []

    filas = []
    for item in info.get('entries', []) or []:
        if not item:
            continue
        video_id = item.get('id')
        if not video_id:
            continue
        filas.append({
            'video_id': video_id,
            'url': f'https://www.youtube.com/watch?v={video_id}',
            'title': item.get('title'),
            'upload_date': item.get('upload_date'),
            'duration': item.get('duration'),
            'view_count': item.get('view_count'),
            'channel_url': channel_url,
        })
    return filas


In [9]:
MAX_VIDEOS_POR_CANAL = 25

videos = []
for _, canal in tqdm(canales_semilla.iterrows(), total=len(canales_semilla)):
    try:
        filas = listar_videos_canal(canal['url'], max_videos=MAX_VIDEOS_POR_CANAL)
        for fila in filas:
            fila['channel_title'] = canal['nombre']
            fila['categoria_fuente'] = canal['categoria']
        videos.extend(filas)
    except Exception as exc:
        print('No se pudo leer canal:', canal['nombre'], exc)

videos_df = pd.DataFrame(videos).drop_duplicates('video_id')
videos_df.to_csv(RAW_DIR / 'videos_candidatos.csv', index=False)
videos_df.head()

  0%|          | 0/29 [00:00<?, ?it/s]

ERROR: [youtube:tab] @exitosanoticias: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @canaln: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @PanamericanaTV: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @ElCacas: This channel does not have a videos tab
ERROR: [youtube:tab] @NegroFuertes: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @JasonQqqOficial: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @LaCotorrisaPeru: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @MagalyTVLaFirme: This channel does not have a videos tab
ERROR: [youtube:tab] @AmoryFuego: This channel does not have a vi

,video_id,url,title,upload_date,duration,view_count,channel_url,channel_title,categoria_fuente
0,TpVbEyV7xS0,https://www.youtube.com/watch?v=TpVbEyV7xS0,I Like 💖| Ocram highlights #2,None,63.0,None,https://www.youtube.com/@ocram,Marco Sifuentes / Ocram,politica_analisis
1,dxsSfcrOE6o,https://www.youtube.com/watch?v=dxsSfcrOE6o,Walk 🚶| Ocram Highlights #1,None,86.0,None,https://www.youtube.com/@ocram,Marco Sifuentes / Ocram,politica_analisis
2,doBMCJT5Y58,https://www.youtube.com/watch?v=doBMCJT5Y58,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,3006.0,None,https://www.youtube.com/@curwen,El diario de Curwen,politica_opinion
3,LFKGSJKX7TY,https://www.youtube.com/watch?v=LFKGSJKX7TY,POLITICAL BRUTALITY AT THE TEATRO CANOUT - ANT...,None,6508.0,None,https://www.youtube.com/@curwen,El diario de Curwen,politica_opinion
4,BoI9o3qpgsQ,https://www.youtube.com/watch?v=BoI9o3qpgsQ,MESIAS GUEVARA ON POLITICAL BRUTALITY | #POLIT...,None,3798.0,None,https://www.youtube.com/@curwen,El diario de Curwen,politica_opinion


## 5. Descarga de subtitulos publicos

Se descargan subtitulos en espanol cuando existan. Si un video no tiene subtitulos, queda fuera del corpus inicial o se transcribe luego con un modelo ASR local.

In [10]:
SUBS_DIR = RAW_DIR / 'subtitulos'
SUBS_DIR.mkdir(parents=True, exist_ok=True)


def descargar_subtitulos(video_url, video_id):
    """Descarga subtítulos en español (manuales o auto-generados).
    Devuelve el Path al .vtt o None si no hay subtítulos disponibles.
    """
    outtmpl = str(SUBS_DIR / f'{video_id}.%(ext)s')
    opciones = {
        **_YT_BASE_OPTS,
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['es', 'es-419', 'es-PE'],
        'subtitlesformat': 'vtt',
        'outtmpl': outtmpl,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            ydl.download([video_url])
    except Exception:
        pass
    archivos = sorted(SUBS_DIR.glob(f'{video_id}*.vtt'))
    return archivos[0] if archivos else None


def descargar_subtitulos_transcript_api(video_id):
    """Fallback: obtiene transcripción vía youtube-transcript-api.
    El parche ssl del inicio cubre también las requests de esta librería.
    """
    try:
        from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        try:
            t = transcript_list.find_manually_created_transcript(['es', 'es-419', 'es-PE'])
        except NoTranscriptFound:
            t = transcript_list.find_generated_transcript(['es', 'es-419', 'es-PE'])
        return [{'start': s['start'], 'duration': s['duration'], 'text': s['text']}
                for s in t.fetch()]
    except Exception:
        return []


def tiempo_a_segundos(valor):
    partes = valor.replace(',', '.').split(':')
    partes = [float(p) for p in partes]
    if len(partes) == 3:
        h, m, s = partes
    else:
        h, m, s = 0, partes[0], partes[1]
    return h * 3600 + m * 60 + s


def limpiar_linea_vtt(linea):
    linea = re.sub(r'<[^>]+>', '', linea)
    linea = re.sub(r'&amp;', '&', linea)
    linea = re.sub(r'&nbsp;', ' ', linea)
    linea = re.sub(r'\s+', ' ', linea).strip()
    return linea


def leer_vtt(path):
    texto = path.read_text(encoding='utf-8', errors='ignore')
    bloques = re.split(r'\n\s*\n', texto)
    segmentos = []
    vistos = set()
    patron_tiempo = re.compile(
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
        r'\s+-->\s+'
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
    )
    for bloque in bloques:
        lineas = [ln.strip() for ln in bloque.splitlines() if ln.strip()]
        if not lineas:
            continue
        match = None
        idx = 0
        for i, linea in enumerate(lineas):
            match = patron_tiempo.search(linea)
            if match:
                idx = i
                break
        if not match:
            continue
        start = tiempo_a_segundos(match.group(1))
        end = tiempo_a_segundos(match.group(2))
        frase = ' '.join(limpiar_linea_vtt(ln) for ln in lineas[idx + 1:])
        frase = re.sub(r'\s+', ' ', frase).strip()
        clave = (round(start, 1), frase.lower())
        if frase and clave not in vistos:
            vistos.add(clave)
            segmentos.append({'start': start, 'duration': max(end - start, 0.1), 'text': frase})
    return segmentos


In [11]:
def hash_texto(texto):
    return hashlib.md5(texto.encode('utf-8')).hexdigest()


MIN_VTT_BYTES = 800   # un .vtt vacío o truncado pesa menos; uno real suele ser varios KB


def vtt_valido(path):
    """True si el archivo existe y supera el tamaño mínimo esperado."""
    return path.exists() and path.stat().st_size >= MIN_VTT_BYTES


# Cargar transcripciones ya guardadas para no reprocesar videos existentes
salida = RAW_DIR / 'transcripts_raw.jsonl'
ids_ya_procesados = set()
transcripciones = []
if salida.exists():
    with open(salida, encoding='utf-8') as f:
        for linea in f:
            row = json.loads(linea)
            transcripciones.append(row)
            ids_ya_procesados.add(row['video_id'])
    print(f'Transcripciones previas cargadas: {len(transcripciones)}')

sin_subs = []
pendientes = videos_df[~videos_df['video_id'].isin(ids_ya_procesados)]
print(f'Videos pendientes de procesar   : {len(pendientes)} / {len(videos_df)}')

barra = tqdm(pendientes.iterrows(), total=len(pendientes), desc='Subtítulos', unit='video')
for _, video in barra:
    vid = video['video_id']
    segmentos = []
    fuente_subs = None

    # ── Verificar si ya existe un .vtt válido en disco ───────────────────────
    archivos_existentes = [p for p in sorted(SUBS_DIR.glob(f'{vid}*.vtt')) if vtt_valido(p)]
    if archivos_existentes:
        segmentos = leer_vtt(archivos_existentes[0])
        fuente_subs = 'yt-dlp-vtt (cache)'
    else:
        # --- Intento 1: yt-dlp (subtítulos del archivo .vtt) ---
        try:
            vtt_path = descargar_subtitulos(video['url'], vid)
            if vtt_path and vtt_valido(vtt_path):
                segmentos = leer_vtt(vtt_path)
                fuente_subs = 'yt-dlp-vtt'
        except Exception as exc:
            tqdm.write(f'  yt-dlp falló en {vid}: {exc}')

        # --- Intento 2: youtube-transcript-api como fallback ---
        if not segmentos:
            segmentos = descargar_subtitulos_transcript_api(vid)
            if segmentos:
                fuente_subs = 'transcript-api'

    texto = ' '.join(seg['text'] for seg in segmentos)
    if len(texto) < 200:
        sin_subs.append(vid)
        barra.set_postfix(ok=len(transcripciones), sin_subs=len(sin_subs), fuente='—')
        continue

    transcripciones.append({
        'video_id': vid,
        'url': video['url'],
        'title': video.get('title'),
        'channel_title': video.get('channel_title'),
        'categoria_fuente': video.get('categoria_fuente'),
        'fuente_subs': fuente_subs,
        'text_hash': hash_texto(texto),
        'segments': segmentos,
    })
    barra.set_postfix(ok=len(transcripciones), sin_subs=len(sin_subs), fuente=fuente_subs or '—')

# Reescribir el JSONL completo (previas + nuevas)
with open(salida, 'w', encoding='utf-8') as f:
    for row in transcripciones:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'\nTranscripciones totales           : {len(transcripciones)}')
print(f'  - previas (omitidas)            : {len(ids_ya_procesados)}')
print(f'  - nuevas en esta ejecución      : {len(transcripciones) - len(ids_ya_procesados)}')
print(f'Sin subtítulos (omitidos)         : {len(sin_subs)}')
tasa = len(transcripciones) / max(len(videos_df), 1) * 100
print(f'Tasa de cobertura                 : {tasa:.1f}%')
print(f'Archivo: {salida}')


Transcripciones previas cargadas: 316
Videos pendientes de procesar   : 16 / 331


Subtítulos:   0%|          | 0/16 [00:00<?, ?video/s]

ERROR: [youtube] LFKGSJKX7TY: Join this channel to get access to members-only content like this video, and other exclusive perks.
ERROR: [youtube] AumPu5lPMp8: Join this channel to get access to members-only content like this video, and other exclusive perks.


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: Did not get any data blocks


ERROR: [youtube] GM3QdzndLLw: This video is not available
ERROR: [youtube] 8Xlz2EelLDA: This video is not available



Transcripciones totales           : 317
  - previas (omitidas)            : 316
  - nuevas en esta ejecución      : 1
Sin subtítulos (omitidos)         : 15
Tasa de cobertura                 : 95.8%
Archivo: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\transcripts_raw.jsonl


## 6. Descarga de videos

Descarga el archivo de video (audio + video) para los videos que ya tienen transcripción confirmada. Ajusta `MAX_VIDEOS_DESCARGA` según el espacio disponible en disco. La calidad por defecto es 720p; cambiar a `'best'` para máxima resolución.

In [18]:
# ── Descarga de videos con yt-dlp ────────────────────────────────────────────
VIDEO_DIR = RAW_DIR / 'videos'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

MAX_VIDEOS_DESCARGA = 300          # ← ajustar según espacio disponible en disco
CALIDAD_VIDEO       = 'bestvideo[height<=720]+bestaudio/best[height<=720]'
MIN_MP4_BYTES       = 500_000    # 500 KB; cualquier video real supera este umbral


def mp4_valido(path):
    """True si el archivo existe y supera el tamaño mínimo esperado."""
    return path.exists() and path.stat().st_size >= MIN_MP4_BYTES


def descargar_video(video_url, video_id, calidad=CALIDAD_VIDEO):
    """Descarga audio+video con la calidad indicada; retorna Path o None.
    Si ya existe un .mp4 válido (>= MIN_MP4_BYTES), lo retorna sin re-descargar.
    """
    existentes = [p for p in sorted(VIDEO_DIR.glob(f'{video_id}*.mp4')) if mp4_valido(p)]
    if existentes:
        return existentes[0]          # ya descargado y verificado

    outtmpl = str(VIDEO_DIR / f'{video_id}.%(ext)s')
    opciones = {
        **_YT_BASE_OPTS,
        'format': calidad,
        'outtmpl': outtmpl,
        'merge_output_format': 'mp4',
        'quiet': False,
        'no_warnings': True,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            ydl.download([video_url])
    except Exception as exc:
        tqdm.write(f'  ✗ Error descargando {video_id}: {exc}')
        return None

    resultado = [p for p in sorted(VIDEO_DIR.glob(f'{video_id}*.mp4')) if mp4_valido(p)]
    if not resultado:
        tqdm.write(f'  ✗ Archivo descargado no supera tamaño mínimo ({MIN_MP4_BYTES} B): {video_id}')
        return None
    return resultado[0]


# Seleccionar videos a descargar (preferencia: los que ya tienen transcripción)
ids_con_transcript = {t['video_id'] for t in transcripciones}
videos_a_descargar = (
    videos_df[videos_df['video_id'].isin(ids_con_transcript)]
    .head(MAX_VIDEOS_DESCARGA)
)

descargados_nuevos = []
ya_existentes      = []
errores_video      = []

barra_v = tqdm(
    videos_a_descargar.iterrows(),
    total=len(videos_a_descargar),
    desc='Descargando videos',
    unit='video',
)
for _, video in barra_v:
    vid = video['video_id']
    barra_v.set_postfix(id=vid[:11], nuevo=len(descargados_nuevos),
                        cache=len(ya_existentes), err=len(errores_video))

    # Verificar si ya existe un archivo válido antes de llamar a yt-dlp
    if any(mp4_valido(p) for p in VIDEO_DIR.glob(f'{vid}*.mp4')):
        ya_existentes.append(vid)
        continue

    path = descargar_video(video['url'], vid)
    if path:
        descargados_nuevos.append({'video_id': vid, 'path': str(path), 'title': video.get('title')})
    else:
        errores_video.append(vid)

barra_v.set_postfix(nuevo=len(descargados_nuevos), cache=len(ya_existentes), err=len(errores_video))
print(f'\nVideos nuevos descargados  : {len(descargados_nuevos)}')
print(f'Ya existían y válidos      : {len(ya_existentes)}')
print(f'Errores / incompletos      : {len(errores_video)}')
tasa_v = (len(descargados_nuevos) + len(ya_existentes)) / max(len(videos_a_descargar), 1) * 100
print(f'Cobertura total            : {tasa_v:.1f}%')
print(f'Directorio                 : {VIDEO_DIR}')


Descargando videos:   0%|          | 0/300 [00:00<?, ?video/s]

[youtube] Extracting URL: https://www.youtube.com/watch?v=3oxwgvWZnsg
[youtube] 3oxwgvWZnsg: Downloading webpage
[youtube] 3oxwgvWZnsg: Downloading webpage
[youtube] 3oxwgvWZnsg: Downloading android vr player API JSON
[info] 3oxwgvWZnsg: Downloading 1 format(s): 398+251
[download] Sleeping 1.74 seconds ...
[download] Destination: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\videos\3oxwgvWZnsg.f398.mp4
[download] 100% of   48.85MiB in 00:00:05 at 8.25MiB/s     
[download] Sleeping 2.68 seconds ...
[download] Destination: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\videos\3oxwgvWZnsg.f251.webm
[download] 100% of    9.53MiB in 00:00:00 at 11.51MiB/s    
[Merger] Merging formats into "D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\videos\3oxwgvWZnsg.mp4"
Deleting original file D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\videos\3oxwgvWZnsg.f398.mp4 (pass -k to keep)
Deleting original file D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\videos\3oxwgvWZnsg.f251.webm (pass -k to ke

## 7. Revision rapida

Antes de pasar al cuaderno 02, se recomienda revisar manualmente canales, videos y transcripciones para retirar contenido con audio deficiente, publicidad extensa o segmentos que no pertenecen al objetivo del corpus.

In [19]:
resumen = pd.DataFrame([
    {'archivo': 'canales_semilla.csv', 'ruta': str(RAW_DIR / 'canales_semilla.csv')},
    {'archivo': 'videos_candidatos.csv', 'ruta': str(RAW_DIR / 'videos_candidatos.csv')},
    {'archivo': 'transcripts_raw.jsonl', 'ruta': str(RAW_DIR / 'transcripts_raw.jsonl')},
])
resumen

,archivo,ruta
0,canales_semilla.csv,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...
1,videos_candidatos.csv,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...
2,transcripts_raw.jsonl,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...


In [20]:
# ── Estadísticas finales: videos por canal × transcripción × video descargado ─
ids_con_transcript = {t['video_id'] for t in transcripciones}
ids_con_video      = {p.stem for p in VIDEO_DIR.glob('*.mp4') if mp4_valido(p)}

stat = (
    videos_df
    .assign(
        transcript = videos_df['video_id'].isin(ids_con_transcript),
        video_mp4  = videos_df['video_id'].isin(ids_con_video),
    )
    .groupby('channel_title', sort=False)
    .agg(
        listados    = ('video_id',    'count'),
        transcript  = ('transcript',  'sum'),
        video_mp4   = ('video_mp4',   'sum'),
    )
    .reset_index()
    .sort_values('transcript', ascending=False)
    .rename(columns={'channel_title': 'canal'})
)

stat['% transcript'] = (stat['transcript'] / stat['listados'] * 100).round(1)
stat['% video']      = (stat['video_mp4']  / stat['listados'] * 100).round(1)

# ── Totales ───────────────────────────────────────────────────────────────────
tot_l = stat['listados'].sum()
tot_t = stat['transcript'].sum()
tot_v = stat['video_mp4'].sum()

COL = 32
print(f"\n{'Canal':<{COL}} {'Listados':>8}  {'Transcript':>10} {'%':>6}  {'Video MP4':>9} {'%':>6}")
print('─' * (COL + 46))
for _, r in stat.iterrows():
    print(f"{r['canal'][:COL-1]:<{COL}} {r['listados']:>8}  "
          f"{r['transcript']:>10} {r['% transcript']:>5.1f}%  "
          f"{r['video_mp4']:>9} {r['% video']:>5.1f}%")
print('─' * (COL + 46))
print(f"{'TOTAL':<{COL}} {tot_l:>8}  "
      f"{tot_t:>10} {tot_t/max(tot_l,1)*100:>5.1f}%  "
      f"{tot_v:>9} {tot_v/max(tot_l,1)*100:>5.1f}%")

# ── Tabla interactiva ─────────────────────────────────────────────────────────
stat.style.bar(subset=['% transcript','% video'], color='#5fba7d', vmin=0, vmax=100)



Canal                            Listados  Transcript      %  Video MP4      %
──────────────────────────────────────────────────────────────────────────────
Sin Guion con Rosa Maria Palaci        25          25 100.0%         25 100.0%
Willax Television                      25          25 100.0%         25 100.0%
RPP Noticias                           25          25 100.0%         25 100.0%
ATV Noticias                           25          25 100.0%         25 100.0%
Latina Noticias                        25          25 100.0%         25 100.0%
Todo Good                              25          25 100.0%         25 100.0%
Hablando Huevadas                      25          25 100.0%         25 100.0%
Viaja y Prueba                         25          25 100.0%          9  36.0%
Misias pero viajeras                   25          25 100.0%         25 100.0%
Goblinciano                            25          25 100.0%         25 100.0%
El diario de Curwen                    25          

,canal,listados,transcript,video_mp4,% transcript,% video
2,Sin Guion con Rosa Maria Palacios,25,25,25,100.000000,100.000000
4,Willax Television,25,25,25,100.000000,100.000000
3,RPP Noticias,25,25,25,100.000000,100.000000
5,ATV Noticias,25,25,25,100.000000,100.000000
6,Latina Noticias,25,25,25,100.000000,100.000000
8,Todo Good,25,25,25,100.000000,100.000000
7,Hablando Huevadas,25,25,25,100.000000,100.000000
14,Viaja y Prueba,25,25,9,100.000000,36.000000
12,Misias pero viajeras,25,25,25,100.000000,100.000000
9,Goblinciano,25,25,25,100.000000,100.000000
